In [19]:
!pip install anthropic beautifulsoup4 lxml openpyxl
import requests

HEADERS = {"User-Agent": "Matthew DOrsogna matthewdorsogna1@gmail.com"}
r = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS)
print(r.status_code)


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
200


In [20]:
tickers = r.json()
for row in tickers.values():
    if row["ticker"] == "LUV":
        print(row)

{'cik_str': 92380, 'ticker': 'LUV', 'title': 'SOUTHWEST AIRLINES CO'}


In [21]:
cik = "0000092380"
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
facts = requests.get(url, headers=HEADERS).json()

gaap = facts["facts"]["us-gaap"]
print(len(gaap), "line items reported")
print([tag for tag in gaap if "InterestExpense" in tag])

638 line items reported
['FinanceLeaseInterestExpense', 'InterestExpense', 'InterestExpenseDebt', 'InterestExpenseNonoperating']


In [22]:
def annual_value(tag, year="CY2025"):
    rows = gaap[tag]["units"]["USD"]
    for r in rows:
        if r.get("frame") == year:
            return r["val"]
    return None

op_income = annual_value("OperatingIncomeLoss")
interest = annual_value("InterestExpense")

print("Operating income:", op_income)
print("Interest expense:", interest)
if op_income and interest:
    print("Interest coverage:", round(op_income / interest, 2), "x")

Operating income: 428000000
Interest expense: None


In [23]:
for tag in ["InterestExpense", "InterestExpenseDebt", "InterestExpenseNonoperating", "FinanceLeaseInterestExpense"]:
    print(tag, "| 2025:", annual_value(tag), "| 2024:", annual_value(tag, "CY2024"))

InterestExpense | 2025: None | 2024: None
InterestExpenseDebt | 2025: None | 2024: None
InterestExpenseNonoperating | 2025: 167000000 | 2024: 249000000
FinanceLeaseInterestExpense | 2025: 3000000 | 2024: 4000000


In [24]:
import time
import pandas as pd

def get_gaap(ticker):
    cik = next(str(r["cik_str"]).zfill(10) for r in tickers.values() if r["ticker"] == ticker)
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    return requests.get(url, headers=HEADERS).json()["facts"]["us-gaap"]

def value(gaap, tags, year="CY2025"):
    for tag in tags:
        if tag in gaap:
            for r in gaap[tag]["units"].get("USD", []):
                if r.get("frame") == year:
                    return r["val"]
    return None

INTEREST_TAGS = ["InterestExpenseNonoperating", "InterestExpense", "InterestExpenseDebt"]

results = []
for t in ["LUV", "DAL", "UAL", "AAL", "ALK", "JBLU"]:
    gaap = get_gaap(t)
    oi = value(gaap, ["OperatingIncomeLoss"])
    ie = value(gaap, INTEREST_TAGS)
    results.append({
        "Ticker": t,
        "Operating Income ($M)": oi / 1e6 if oi is not None else None,
        "Interest Expense ($M)": ie / 1e6 if ie else None,
        "Coverage (x)": round(oi / ie, 2) if oi is not None and ie else None,
    })
    time.sleep(0.2)

df = pd.DataFrame(results)
df

,Ticker,Operating Income ($M),Interest Expense ($M),Coverage (x)
0,LUV,428.0,167.0,2.56
1,DAL,5822.0,NaN,NaN
2,UAL,4713.0,1373.0,3.43
3,AAL,1467.0,1716.0,0.85
4,ALK,303.0,272.0,1.11
5,JBLU,-368.0,588.0,-0.63


In [25]:
def value(gaap, tags, year="CY2025"):
    for tag in tags:
        if tag in gaap:
            for r in gaap[tag]["units"].get("USD", []):
                if r.get("frame") == year:
                    return r["val"], tag
    return None, None

INTEREST_TAGS = ["InterestExpenseNonoperating", "InterestExpense", "InterestExpenseDebt",
                 "InterestIncomeExpenseNonoperatingNet"]

results = []
for t in ["LUV", "DAL", "UAL", "AAL", "ALK", "JBLU"]:
    gaap = get_gaap(t)
    oi, _ = value(gaap, ["OperatingIncomeLoss"])
    ie, ie_tag = value(gaap, INTEREST_TAGS)
    if ie is not None:
        ie = abs(ie)  # net interest is reported as a negative number
    results.append({
        "Ticker": t,
        "Operating Income ($M)": oi / 1e6 if oi is not None else None,
        "Interest Expense ($M)": ie / 1e6 if ie else None,
        "Coverage (x)": round(oi / ie, 2) if oi is not None and ie else None,
        "Interest Source": ie_tag,
    })
    time.sleep(0.2)

df = pd.DataFrame(results)
df

,Ticker,Operating Income ($M),Interest Expense ($M),Coverage (x),Interest Source
0,LUV,428.0,167.0,2.56,InterestExpenseNonoperating
1,DAL,5822.0,679.0,8.57,InterestIncomeExpenseNonoperatingNet
2,UAL,4713.0,1373.0,3.43,InterestExpenseNonoperating
3,AAL,1467.0,1716.0,0.85,InterestExpenseNonoperating
4,ALK,303.0,272.0,1.11,InterestExpenseNonoperating
5,JBLU,-368.0,588.0,-0.63,InterestExpenseNonoperating


In [26]:
DA_TAGS = ["DepreciationDepletionAndAmortization", "DepreciationAndAmortization",
           "DepreciationAmortizationAndAccretionNet"]
DEBT_LT_TAGS = ["LongTermDebtNoncurrent", "LongTermDebtAndCapitalLeaseObligations"]
DEBT_CUR_TAGS = ["LongTermDebtCurrent", "LongTermDebtAndCapitalLeaseObligationsCurrent"]
CASH_TAGS = ["CashAndCashEquivalentsAtCarryingValue", "Cash"]

YEAR_END = "CY2025Q4I"   # balance sheet numbers are a snapshot on Dec 31

rows = []
for t in df["Ticker"]:
    gaap = get_gaap(t)
    oi, _ = value(gaap, ["OperatingIncomeLoss"])
    da, _ = value(gaap, DA_TAGS)
    debt_lt, _ = value(gaap, DEBT_LT_TAGS, YEAR_END)
    debt_cur, _ = value(gaap, DEBT_CUR_TAGS, YEAR_END)
    cash, _ = value(gaap, CASH_TAGS, YEAR_END)

    ebitda = oi + da if oi is not None and da is not None else None
    debt = (debt_lt or 0) + (debt_cur or 0) if debt_lt is not None else None

    rows.append({
        "Ticker": t,
        "EBITDA ($M)": ebitda / 1e6 if ebitda is not None else None,
        "Total Debt ($M)": debt / 1e6 if debt is not None else None,
        "Cash ($M)": cash / 1e6 if cash is not None else None,
        "Debt / EBITDA (x)": round(debt / ebitda, 2) if debt and ebitda else None,
    })
    time.sleep(0.2)

lev = pd.DataFrame(rows)
full = df.merge(lev, on="Ticker")

def flags(r):
    f = []
    if r["Coverage (x)"] is not None and r["Coverage (x)"] < 2:
        f.append("weak coverage")
    if r["Debt / EBITDA (x)"] is not None and r["Debt / EBITDA (x)"] > 4:
        f.append("high leverage")
    return ", ".join(f)

full["Flags"] = full.apply(flags, axis=1)
full

,Ticker,Operating Income ($M),Interest Expense ($M),Coverage (x),Interest Source,EBITDA ($M),Total Debt ($M),Cash ($M),Debt / EBITDA (x),Flags
0,LUV,428.0,167.0,2.56,InterestExpenseNonoperating,1988.0,4901.0,3231.0,2.47,
1,DAL,5822.0,679.0,8.57,InterestIncomeExpenseNonoperatingNet,8265.0,13308.0,4310.0,1.61,
2,UAL,4713.0,1373.0,3.43,InterestExpenseNonoperating,7652.0,21266.0,5942.0,2.78,
3,AAL,1467.0,1716.0,0.85,InterestExpenseNonoperating,3667.0,28895.0,954.0,7.88,"weak coverage, high leverage"
4,ALK,303.0,272.0,1.11,InterestExpenseNonoperating,1098.0,5309.0,627.0,4.84,"weak coverage, high leverage"
5,JBLU,-368.0,588.0,-0.63,InterestExpenseNonoperating,320.0,8498.0,1946.0,26.56,"weak coverage, high leverage"


In [27]:
import os
import getpass
import anthropic

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Paste your API key: ")
client = anthropic.Anthropic()

msg = client.messages.create(
    model="claude-sonnet-5-5",
    max_tokens=100,
    messages=[{"role": "user", "content": "In one sentence, what does interest coverage measure?"}],
)
print(msg.content[0].text)

Paste your API key: ******
Interest coverage measures how easily a company can pay the interest on its outstanding debt, typically calculated as earnings before interest and taxes (EBIT) divided by interest expense.


In [28]:
import re
from bs4 import BeautifulSoup

def latest_10k_text(ticker):
    cik = next(str(r["cik_str"]).zfill(10) for r in tickers.values() if r["ticker"] == ticker)
    sub = requests.get(f"https://data.sec.gov/submissions/CIK{cik}.json", headers=HEADERS).json()
    rec = sub["filings"]["recent"]
    for form, acc, doc in zip(rec["form"], rec["accessionNumber"], rec["primaryDocument"]):
        if form == "10-K":
            url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc.replace('-', '')}/{doc}"
            html = requests.get(url, headers=HEADERS).content
            text = BeautifulSoup(html, "lxml").get_text(" ")
            return re.sub(r"\s+", " ", text), url
    return None, None

def best_section(text, heading, length=40000):
    best, best_score = "", -1
    for m in re.finditer(heading, text, flags=re.I):
        chunk = text[m.start(): m.start() + length]
        score = len(re.findall(r"\bdebt\b|notes due|credit facility|covenant|maturit", chunk, flags=re.I))
        if score > best_score:
            best, best_score = chunk, score
    return best

luv_text, luv_url = latest_10k_text("LUV")
print(luv_url)
print(len(luv_text), "characters in the full 10-K")

luv_liq = best_section(luv_text, "Liquidity and Capital Resources")
print(luv_liq[:1500])

https://www.sec.gov/Archives/edgar/data/92380/000009238026000004/luv-20251231.htm
571321 characters in the full 10-K
Liquidity and Capital Resources Net cash provided by operating activities for 2025 was $1.8 billion, and net cash provided by operating activities for 2024 was $462 million. Operating cash inflows are historically primarily derived from selling tickets and providing air transportation to Customers. The vast majority of tickets are purchased prior to the day on which travel is provided and, in some cases, several months before the anticipated travel date. Operating cash outflows are related to the recurring expenses of airline operations. The operating cash flows for 2025 were largely impacted by the Company's net results (as adjusted for noncash items), a $103 million profit-sharing contribution for 2024 pursuant to the Company's Retirement Savings Plan, and a $1.1 billion decrease in Air traffic liability primarily driven by immediate recognition of a larger portion of 

In [29]:
import json

def ask_claude(prompt):
    msg = client.messages.create(
        model="claude-sonnet-5-5",
        max_tokens=4000,
        messages=[{"role": "user", "content": prompt}],
    )
    raw = "".join(block.text for block in msg.content if block.type == "text")
    match = re.search(r"\{.*\}", raw, re.S)
    return json.loads(match.group(0)) if match else {}

def credit_summary(ticker, section):
    prompt = f"""You are a credit analyst. From this 10-K excerpt for {ticker}, return ONLY a JSON object with these keys:
"debt_maturities_next_2yrs": one short sentence on debt coming due soon,
"financial_covenants": one short sentence on covenants (or "none mentioned"),
"liquidity_sources": one short sentence on cash, credit lines, and other liquidity,
"top_credit_risks": a list of 3 short strings.
Use only what the excerpt says.

EXCERPT:
{section}"""
    return ask_claude(prompt)

luv_summary = credit_summary("LUV", luv_liq)
for key, val in luv_summary.items():
    print(key, "->", val)

debt_maturities_next_2yrs -> The excerpt gives no full maturity schedule (it points to Note 6), but about $426 million of remaining payroll support loans must be repaid to Treasury in future periods, and fixed-rate notes include $100 million of 7.375% debentures due 2027 and $750 million of 4.375% notes due 2028.
financial_covenants -> The Amended Credit Agreement requires a Coverage Ratio of at least 1.25 to 1.00 (with a one-time option to reduce it to 0.80 to 1.00 for two consecutive quarters); the company was in compliance at December 31, 2025, and there are also credit rating triggers in card processing and hedging agreements.
liquidity_sources -> Total available liquidity was $4.7 billion at year-end 2025, made up of $3.2 billion of cash and cash equivalents and an undrawn $1.5 billion revolver expiring August 2028, plus about $17.0 billion of unencumbered aircraft and other assets, investment-grade ratings (Baa2/BBB/BBB+), an effective shelf registration, and operating cash flow 

In [30]:
def income_section(text):
    best, best_score = "", -1
    for m in re.finditer(r"Consolidated Statements? of (Operations|Income)|Statements? of Consolidated (Operations|Income)", text, flags=re.I):
        chunk = text[m.start(): m.start() + 8000]
        score = len(re.findall(r"\d{1,3}(?:,\d{3})+", chunk))
        if score > best_score:
            best, best_score = chunk, score
    return best

def ai_income(text):
    prompt = f"""From this consolidated income statement excerpt, use the MOST RECENT fiscal year column.
Return ONLY a JSON object:
{{"revenue": total operating revenues as shown (number, no commas),
  "operating_income": operating income as shown (number, negative if a loss),
  "units": "millions", "thousands", or "dollars" based on the statement header}}

EXCERPT:
{income_section(text)}"""
    d = ask_claude(prompt)
    mult = {"millions": 1e6, "thousands": 1e3, "dollars": 1}.get(str(d.get("units", "")).lower(), 1e6)
    try:
        return float(d["revenue"]) * mult, float(d["operating_income"]) * mult
    except (KeyError, TypeError, ValueError):
        return None, None

ai_rows, check_rows = [], []
for t in full["Ticker"]:
    print("Reading", t, "10-K...")
    text, url = latest_10k_text(t)

    s = credit_summary(t, best_section(text, "Liquidity and Capital Resources"))
    risks = s.get("top_credit_risks", [])
    ai_rows.append({
        "Ticker": t,
        "Debt Maturities": s.get("debt_maturities_next_2yrs"),
        "Covenants": s.get("financial_covenants"),
        "Liquidity": s.get("liquidity_sources"),
        "Top Credit Risks": "; ".join(risks) if isinstance(risks, list) else risks,
        "10-K Link": url,
    })

    ai_rev, ai_oi = ai_income(text)
    gaap = get_gaap(t)
    sec_rev, _ = value(gaap, ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"])
    sec_oi, _ = value(gaap, ["OperatingIncomeLoss"])
    for label, ai_val, sec_val in [("Revenue", ai_rev, sec_rev), ("Operating Income", ai_oi, sec_oi)]:
        diff = round(abs(ai_val - sec_val) / abs(sec_val) * 100, 2) if ai_val is not None and sec_val else None
        check_rows.append({
            "Ticker": t, "Metric": label,
            "AI Extracted ($M)": ai_val / 1e6 if ai_val is not None else None,
            "SEC Reported ($M)": sec_val / 1e6 if sec_val is not None else None,
            "Difference (%)": diff,
            "Result": "match" if diff is not None and diff < 1 else "CHECK",
        })
    time.sleep(0.2)

ai_df = pd.DataFrame(ai_rows)
check_df = pd.DataFrame(check_rows)

with pd.ExcelWriter("10k_analysis.xlsx") as xl:
    full.to_excel(xl, sheet_name="Credit Ratios", index=False)
    ai_df.to_excel(xl, sheet_name="AI Credit Summary", index=False)
    check_df.to_excel(xl, sheet_name="AI Validation", index=False)

print("\nSaved 10k_analysis.xlsx")
check_df

Reading LUV 10-K...
Reading DAL 10-K...
Reading UAL 10-K...
Reading AAL 10-K...
Reading ALK 10-K...
Reading JBLU 10-K...

Saved 10k_analysis.xlsx


,Ticker,Metric,AI Extracted ($M),SEC Reported ($M),Difference (%),Result
0,LUV,Revenue,28063.0,28063.0,0.00,match
1,LUV,Operating Income,428.0,428.0,0.00,match
2,DAL,Revenue,63364.0,63364.0,0.00,match
3,DAL,Operating Income,5822.0,5822.0,0.00,match
4,UAL,Revenue,59070.0,59070.0,0.00,match
5,UAL,Operating Income,4713.0,4713.0,0.00,match
6,AAL,Revenue,54626.0,54633.0,0.01,match
7,AAL,Operating Income,1511.0,1467.0,3.00,CHECK
8,ALK,Revenue,14239.0,14239.0,0.00,match
9,ALK,Operating Income,303.0,303.0,0.00,match
